## Reading Flirtflip Dataset from huggingface

In [25]:

import pandas as pd

url = "https://huggingface.co/datasets/shirshatzman/flirtflip-dataset/resolve/01fe856db82764f81a60ee5a94f470aebda69e2d/flirtflip_dataset_1500.csv"

df_flirtflip = pd.read_csv(url)

print("Dataset shape:", df_flirtflip.shape)
print(df_flirtflip.head())


Dataset shape: (1085, 6)
                original                                             gentle  \
0          Great choice!  Great choice! You clearly have excellent taste...   
1    Is this seat taken?     Is this seat taken? I promise I'm good company   
2     Nice weather today  Beautiful day, isn't it? Though I have to say,...   
3  What are you reading?  That book looks fascinating... though not as f...   
4        Long line today  Long line, but at least it gave me time to wor...   

                                             playful  \
0  Great choice! Though between you and me, the s...   
1  Is this seat taken, or were you saving it for ...   
2  Nice weather, right? Or did you just bring the...   
3  Good book? Because I'm writing a story in my h...   
4  This line is taking forever... good thing I ha...   

                                                bold          id  \
0  Great choice! Now make another one and give me...  flirt_0001   
1  This seat's about to be 

In [26]:
## Here original is considered as non flirtation, gentle, playful and bold considered as flirtation

In [27]:

import pandas as pd

# Convert the four message columns into one message column
df_flirtflip_long = df_flirtflip.melt(
    id_vars=["id", "scenario"],
    value_vars=["original", "gentle", "playful", "bold"],
    var_name="style",
    value_name="message"
)

# Assign proposed binary labels
df_flirtflip_long["label"] = (
    df_flirtflip_long["style"] != "original"
).astype(int)

# Remove missing or empty messages
df_flirtflip_long["message"] = (
    df_flirtflip_long["message"].astype("string").str.strip()
)

df_flirtflip_long = df_flirtflip_long.dropna(
    subset=["message"]
)

df_flirtflip_long = df_flirtflip_long[
    df_flirtflip_long["message"] != ""
].reset_index(drop=True)

# Display the new table
display(df_flirtflip_long.head(20))

print("Shape:", df_flirtflip_long.shape)
print("\nLabel distribution:")
print(df_flirtflip_long["label"].value_counts())


,id,scenario,style,message,label
0,flirt_0001,at a dog park,original,Great choice!,0
1,flirt_0002,at the library,original,Is this seat taken?,0
2,flirt_0003,at a food truck,original,Nice weather today,0
3,flirt_0004,at a party,original,What are you reading?,0
4,flirt_0005,at a yoga class,original,Long line today,0
5,flirt_0006,at a networking event,original,They make the best coffee here,0
6,flirt_0007,at a club,original,What's good here?,0
7,flirt_0008,in line,original,Excuse me,0
8,flirt_0009,at the beach,original,Can I get your WiFi password?,0
9,flirt_0010,at a garage sale,original,This place is packed,0


Shape: (4340, 5)

Label distribution:
label
1    3255
0    1085
Name: count, dtype: int64


In [28]:
# we have different versions of messages for same scenario and we don't want the messages belong to the same scenario doesn't appear in test and validation set
# so we used Groupshufflesplit to split the data
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df_flirtflip_long,
        groups=df_flirtflip_long["id"]
    )
)

flirtflip_train = df_flirtflip_long.iloc[train_idx].copy()
flirtflip_test = df_flirtflip_long.iloc[test_idx].copy()

print("FlirtFlip training:", flirtflip_train.shape)
print("FlirtFlip testing:", flirtflip_test.shape)

FlirtFlip training: (3472, 5)
FlirtFlip testing: (868, 5)


In [29]:
# we are verifying that the same scenarios are not present in training and testing
# ensuring there is no overlapping ids
print("FlirtFlip training labels:")
print(flirtflip_train["label"].value_counts().sort_index())

print("\nFlirtFlip testing labels:")
print(flirtflip_test["label"].value_counts().sort_index())

print("\nUnique IDs in training:", flirtflip_train["id"].nunique())
print("Unique IDs in testing:", flirtflip_test["id"].nunique())

print(
    "\nID overlap:",
    len(set(flirtflip_train["id"]) & set(flirtflip_test["id"]))
)


FlirtFlip training labels:
label
0     868
1    2604
Name: count, dtype: int64

FlirtFlip testing labels:
label
0    217
1    651
Name: count, dtype: int64

Unique IDs in training: 868
Unique IDs in testing: 217

ID overlap: 0


In [30]:
## Reading the data from flirting_rated csv

In [31]:
df=pd.read_csv("flirting_rated.csv", index_col=None)

In [32]:
#Remove rows with missing labels
df_clean=df.dropna(subset=["polarity"]).copy()
print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", df_clean.shape)
print("Missing values after cleaning:")
print(df_clean.isnull().sum())

Original dataset shape: (2886, 2)
Cleaned dataset shape: (2873, 2)
Missing values after cleaning:
polarity          0
final_messages    0
dtype: int64


In [33]:
# Test and train split 
from sklearn.model_selection import train_test_split
X=df_clean["final_messages"].astype(str)
Y=df_clean["polarity"].astype(int)
X_train,X_test,Y_train,Y_test=train_test_split(X, Y,
                                               test_size=0.20,
                                               random_state=42,
                                               stratify=Y)
print("Training size:",len(X_train))
print("Testing size:", len(X_test))
print((Y_train.value_counts(normalize=True) * 100).round(2))

Training size: 2298
Testing size: 575
polarity
0    82.29
1    17.71
Name: proportion, dtype: float64


In [36]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
import pandas as pd

# Split the existing FlirtFlip training data into train and validation
splitter_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter_val.split(
        flirtflip_train,
        groups=flirtflip_train["id"]
    )
)

flirtflip_train_final = flirtflip_train.iloc[train_idx].copy()
flirtflip_val = flirtflip_train.iloc[val_idx].copy()

# Split Tinder training data into train and validation
X_tinder_train, X_tinder_val, y_tinder_train, y_tinder_val = (
    train_test_split(
        X_train,
        Y_train,  
        test_size=0.20,
        random_state=42,
        stratify=Y_train
    )
)

## Combining the training data of both the dataset

In [37]:
# Prepare Tinder training data
tinder_train = pd.DataFrame({
    "message": X_tinder_train.astype(str).values,
    "label": y_tinder_train.astype(int).values
})

# Prepare FlirtFlip training data
flirtflip_train_clean = flirtflip_train_final[
    ["message", "label"]
].copy()

flirtflip_train_clean["message"] = (
    flirtflip_train_clean["message"]
    .fillna("")
    .astype(str)
    .str.strip()
)

flirtflip_train_clean["label"] = (
    flirtflip_train_clean["label"].astype(int)
)

flirtflip_train_clean = flirtflip_train_clean[
    flirtflip_train_clean["message"] != ""
].copy()

# Combine
combined_train = pd.concat(
    [tinder_train, flirtflip_train_clean],
    ignore_index=True
)

combined_train = combined_train[
    combined_train["message"].str.strip() != ""
].reset_index(drop=True)

print("Combined training shape:", combined_train.shape)
print("\nTraining label counts:")
print(combined_train["label"].value_counts())

Combined training shape: (4614, 2)

Training label counts:
label
1    2408
0    2206
Name: count, dtype: int64


In [40]:
# Prepare Tinder validation data
tinder_val = pd.DataFrame({
    "message": X_tinder_val.astype(str).values,
    "label": y_tinder_val.astype(int).values
})

# Prepare FlirtFlip validation data
flirtflip_val_clean = flirtflip_val[
    ["message", "label"]
].copy()

flirtflip_val_clean["message"] = (
    flirtflip_val_clean["message"]
    .fillna("")
    .astype(str)
    .str.strip()
)

flirtflip_val_clean["label"] = (
    flirtflip_val_clean["label"].astype(int)
)

flirtflip_val_clean = flirtflip_val_clean[
    flirtflip_val_clean["message"] != ""
].copy()

# Combine
combined_val = pd.concat(
    [tinder_val, flirtflip_val_clean],
    ignore_index=True
)

combined_val = combined_val[
    combined_val["message"].str.strip() != ""
].reset_index(drop=True)

print("Combined validation shape:", combined_val.shape)
print("\nValidation label counts:")
print(combined_val["label"].value_counts())

Combined validation shape: (1156, 2)

Validation label counts:
label
1    603
0    553
Name: count, dtype: int64


In [41]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ),
    "Complement Naive Bayes": ComplementNB()
}

results = []
trained_models = {}

X_train_combined = combined_train["message"].fillna("").astype(str)
y_train_combined = combined_train["label"].astype(int)

X_val_combined = combined_val["message"].fillna("").astype(str)
y_val_combined = combined_val["label"].astype(int)

for name, classifier in models.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            sublinear_tf=True
        )),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train_combined, y_train_combined)

    predictions = pipeline.predict(X_val_combined)

    results.append({
        "Algorithm": name,
        "Accuracy": round(
            accuracy_score(y_val_combined, predictions), 3
        ),
        "Flirt_Precision": round(
            precision_score(
                y_val_combined, predictions,
                pos_label=1, zero_division=0
            ), 3
        ),
        "Flirt_Recall": round(
            recall_score(
                y_val_combined, predictions,
                pos_label=1, zero_division=0
            ), 3
        ),
        "Flirt_F1": round(
            f1_score(
                y_val_combined, predictions,
                pos_label=1, zero_division=0
            ), 3
        )
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values(
    "Flirt_F1", ascending=False
).reset_index(drop=True)

results_df


,Algorithm,Accuracy,Flirt_Precision,Flirt_Recall,Flirt_F1
0,Linear SVM,0.928,0.933,0.929,0.931
1,Logistic Regression,0.912,0.911,0.920,0.916
2,Complement Naive Bayes,0.854,0.814,0.934,0.869
3,Multinomial Naive Bayes,0.832,0.778,0.949,0.855


In [43]:
from sklearn.metrics import classification_report

# Prepare source-specific validation data
tinder_val_messages = X_tinder_val.fillna("").astype(str)
tinder_val_labels = y_tinder_val.astype(int)

flirtflip_val_messages = (
    flirtflip_val["message"].fillna("").astype(str)
)
flirtflip_val_labels = flirtflip_val["label"].astype(int)

source_results = []

for name, pipeline in trained_models.items():

    # Tinder validation
    tinder_pred = pipeline.predict(tinder_val_messages)

    source_results.append({
        "Algorithm": name,
        "Dataset": "Tinder Validation",
        "Accuracy": round(
            accuracy_score(tinder_val_labels, tinder_pred), 3
        ),
        "Flirt_Precision": round(
            precision_score(
                tinder_val_labels, tinder_pred,
                zero_division=0
            ), 3
        ),
        "Flirt_Recall": round(
            recall_score(
                tinder_val_labels, tinder_pred,
                zero_division=0
            ), 3
        ),
        "Flirt_F1": round(
            f1_score(
                tinder_val_labels, tinder_pred,
                zero_division=0
            ), 3
        )
    })

    # FlirtFlip validation
    flirtflip_pred = pipeline.predict(flirtflip_val_messages)

    source_results.append({
        "Algorithm": name,
        "Dataset": "FlirtFlip Validation",
        "Accuracy": round(
            accuracy_score(flirtflip_val_labels, flirtflip_pred), 3
        ),
        "Flirt_Precision": round(
            precision_score(
                flirtflip_val_labels, flirtflip_pred,
                zero_division=0
            ), 3
        ),
        "Flirt_Recall": round(
            recall_score(
                flirtflip_val_labels, flirtflip_pred,
                zero_division=0
            ), 3
        ),
        "Flirt_F1": round(
            f1_score(
                flirtflip_val_labels, flirtflip_pred,
                zero_division=0
            ), 3
        )
    })

source_results_df = pd.DataFrame(source_results)
source_results_df.sort_values(
    ["Dataset", "Flirt_F1"],
    ascending=[True, False]
).reset_index(drop=True)

,Algorithm,Dataset,Accuracy,Flirt_Precision,Flirt_Recall,Flirt_F1
0,Linear SVM,FlirtFlip Validation,0.990,0.987,1.000,0.993
1,Logistic Regression,FlirtFlip Validation,0.983,0.978,1.000,0.989
2,Complement Naive Bayes,FlirtFlip Validation,0.917,0.900,1.000,0.947
3,Multinomial Naive Bayes,FlirtFlip Validation,0.904,0.886,1.000,0.940
4,Linear SVM,Tinder Validation,0.835,0.535,0.469,0.500
5,Multinomial Naive Bayes,Tinder Validation,0.724,0.342,0.617,0.441
6,Complement Naive Bayes,Tinder Validation,0.759,0.366,0.506,0.425
7,Logistic Regression,Tinder Validation,0.804,0.440,0.407,0.423


In [45]:
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

comparison_results = []

X_test_clean = X_test.fillna("").astype(str)
y_test_clean = Y_test.astype(int)

for name, model in trained_models.items():
    predictions = model.predict(X_test_clean)

    print(f"\n{'=' * 50}")
    print(name)
    print("=" * 50)

    print("Confusion Matrix:")
    print(confusion_matrix(
        y_test_clean, predictions, labels=[0, 1]
    ))

    print("\nClassification Report:")
    print(classification_report(
        y_test_clean,
        predictions,
        labels=[0, 1],
        target_names=["No Flirtation", "Flirtation"],
        zero_division=0
    ))

    report = classification_report(
        y_test_clean,
        predictions,
        labels=[0, 1],
        target_names=["No Flirtation", "Flirtation"],
        output_dict=True,
        zero_division=0
    )

    comparison_results.append({
        "Algorithm": name,
        "Accuracy": accuracy_score(y_test_clean, predictions),
        "Flirt_Precision": report["Flirtation"]["precision"],
        "Flirt_Recall": report["Flirtation"]["recall"],
        "Flirt_F1": report["Flirtation"]["f1-score"]
    })

comparison_df = pd.DataFrame(comparison_results).round(3)
comparison_df.sort_values(
    "Flirt_F1", ascending=False
).reset_index(drop=True)


Logistic Regression
Confusion Matrix:
[[404  69]
 [ 59  43]]

Classification Report:
               precision    recall  f1-score   support

No Flirtation       0.87      0.85      0.86       473
   Flirtation       0.38      0.42      0.40       102

     accuracy                           0.78       575
    macro avg       0.63      0.64      0.63       575
 weighted avg       0.79      0.78      0.78       575


Multinomial Naive Bayes
Confusion Matrix:
[[351 122]
 [ 35  67]]

Classification Report:
               precision    recall  f1-score   support

No Flirtation       0.91      0.74      0.82       473
   Flirtation       0.35      0.66      0.46       102

     accuracy                           0.73       575
    macro avg       0.63      0.70      0.64       575
 weighted avg       0.81      0.73      0.75       575


Linear SVM
Confusion Matrix:
[[431  42]
 [ 58  44]]

Classification Report:
               precision    recall  f1-score   support

No Flirtation       0.88 

,Algorithm,Accuracy,Flirt_Precision,Flirt_Recall,Flirt_F1
0,Linear SVM,0.826,0.512,0.431,0.468
1,Multinomial Naive Bayes,0.727,0.354,0.657,0.460
2,Complement Naive Bayes,0.753,0.370,0.559,0.445
3,Logistic Regression,0.777,0.384,0.422,0.402


In [47]:
## Creating a baseline model only for Tinder dataset
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# Original Tinder dataset only
X = df_clean["final_messages"].astype(str)
y = df_clean["polarity"].astype(int)

# Re-create the same reproducible train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Fresh baseline model
baseline_tinder_svm = Pipeline([
    ("vectorizer", TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("classifier", LinearSVC(
        class_weight="balanced",
        random_state=42,
        max_iter=5000
    ))
])

# Train only on Tinder training messages
baseline_tinder_svm.fit(X_train, y_train)

print("Fresh Tinder-only baseline trained successfully!")
print("Training messages:", len(X_train))
print("Test messages:", len(X_test))

Fresh Tinder-only baseline trained successfully!
Training messages: 2298
Test messages: 575


In [49]:
## evaluating the model
baseline_predictions = baseline_tinder_svm.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, baseline_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    baseline_predictions,
    target_names=["No Flirtation", "Flirtation"],
    zero_division=0
))

Confusion Matrix:
[[442  31]
 [ 62  40]]

Classification Report:
               precision    recall  f1-score   support

No Flirtation       0.88      0.93      0.90       473
   Flirtation       0.56      0.39      0.46       102

     accuracy                           0.84       575
    macro avg       0.72      0.66      0.68       575
 weighted avg       0.82      0.84      0.83       575



In [48]:
## Loading whatsapp text
import pandas as pd
df=pd.read_csv("mabel.txt",header=None,on_bad_lines='skip',encoding='utf8')
df=df.drop(0)
df.columns=["Date","Chat"]
message=df["Chat"].str.split("-",n=1,expand=True)
df["Time"]=message[0]
message1=message[1].str.split(":",n=1,expand=True)
df["Name"]=message1[0]
df["Chat"]=message1[1]
df=df[["Date","Time","Name","Chat"]]
df

,Date,Time,Name,Chat
1,05/12/19,1:42 pm,Mabel Infoziant,Hi this is Mabel we just spoke
2,05/12/19,1:42 pm,Mabel Infoziant,What’s your full name
3,05/12/19,1:42 pm,AR❤,Ramisha Rani K
4,05/12/19,1:42 pm,Mabel Infoziant,Ok
5,05/12/19,1:42 pm,Mabel Infoziant,ramisharanik@gmail.com
6,05/12/19,1:43 pm,Mabel Infoziant,Your email Id?
7,05/12/19,1:43 pm,AR❤,Yes Mam
8,05/12/19,1:43 pm,Mabel Infoziant,I will send 2 abstracts for u to start working
9,05/12/19,1:43 pm,AR❤,Yeah mam
10,05/12/19,1:43 pm,Mabel Infoziant,Give me the list that u have too


In [52]:
messages = df["Chat"].fillna("").astype(str)

# Keep the baseline separate from the newer models
all_models = {
    "Baseline - Tinder SVM": baseline_tinder_svm,
    "Logistic Regression - Combined": trained_models["Logistic Regression"],
    "Multinomial NB - Combined": trained_models["Multinomial Naive Bayes"],
    "Linear SVM - Combined": trained_models["Linear SVM"],
    "Complement NB - Combined": trained_models["Complement Naive Bayes"]
}

# Store each model's predictions in a separate column
for name, model in all_models.items():
    column = name.replace(" - ", "_").replace(" ", "_")
    df[column] = model.predict(messages)

print("Predictions completed for", len(df), "WhatsApp messages.")

Predictions completed for 50 WhatsApp messages.


In [53]:
## Comparing prediction results for the basimport pandas as pd

import pandas as pd

comparison_rows = []

for name, model in all_models.items():
    column = name.replace(" - ", "_").replace(" ", "_")
    flirt_count = int(df[column].sum())

    comparison_rows.append({
        "Model": name,
        "Total Messages": len(df),
        "Predicted Flirtation": flirt_count,
        "Predicted Non-Flirtation": len(df) - flirt_count,
        "Flirtation Percentage": round(
            flirt_count / len(df) * 100, 2
        )
    })

whatsapp_model_comparison = pd.DataFrame(comparison_rows)
display(whatsapp_model_comparison)

,Model,Total Messages,Predicted Flirtation,Predicted Non-Flirtation,Flirtation Percentage
0,Baseline - Tinder SVM,50,14,36,28.0
1,Logistic Regression - Combined,50,16,34,32.0
2,Multinomial NB - Combined,50,27,23,54.0
3,Linear SVM - Combined,50,17,33,34.0
4,Complement NB - Combined,50,22,28,44.0
